In [3]:
import tensorflow as tf
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import train_test_split

print("TF version:", tf.__version__)
print("Num GPUs:", len(tf.config.list_physical_devices("GPU")))
print(tf.config.list_physical_devices())

TF version: 2.20.0
Num GPUs: 0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models, Input
from tensorflow.keras.layers import Dropout, GRU, Dense, TimeDistributed, Lambda, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [3]:
train_table = pd.read_pickle("train_table.pkl")

X_videos = np.stack(train_table["Frames"].values).astype("float32")
y_labels = train_table["Label"].astype("int8").values
meta_features = train_table[["light_conditions", "weather_cloudy", "weather_rain", "scene_highway", "scene_rural"]].astype("float32").values

In [4]:
from sklearn.utils import shuffle

X_videos, y_labels, meta_features = shuffle(X_videos, y_labels, meta_features, random_state=42)


In [5]:
X_train, X_val, y_train, y_val, meta_train, meta_val = train_test_split(
    X_videos, y_labels, meta_features,
    test_size=0.3, random_state=42, stratify=y_labels, shuffle=True)

print("Training set shape:", X_train.shape, y_train.shape, meta_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape, meta_val.shape)

Training set shape: (1050, 24, 224, 224, 3) (1050,) (1050, 5)
Validation set shape: (450, 24, 224, 224, 3) (450,) (450, 5)


In [6]:
del X_videos, y_labels, meta_features
gc.collect()

0

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils import shuffle

datagen = ImageDataGenerator(
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.05,
    brightness_range=[0.85, 1.15],
    rotation_range=5,
    horizontal_flip=False)

aug_videos = []
aug_labels = []

for vid, label in zip(X_train, y_train):
    seed = np.random.randint(1e6)
    frames = [datagen.random_transform(frame, seed=seed) for frame in vid]
    aug_videos.append(np.array(frames, dtype=np.float32))
    aug_labels.append(label)

aug_videos = np.array(aug_videos, dtype=np.float32)
aug_labels = np.array(aug_labels)

# Combine original + augmented
X_combined = np.concatenate([X_train, aug_videos], axis=0)
y_combined = np.concatenate([y_train, aug_labels], axis=0)
meta_combined = np.concatenate([meta_train, meta_train], axis=0)

# Shuffle again
X_combined, y_combined, meta_combined = shuffle(
    X_combined, y_combined, meta_combined, random_state=42)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils import shuffle

datagen_2 = ImageDataGenerator(
    width_shift_range=0.075,
    height_shift_range=0.075,
    zoom_range=0.075,
    brightness_range=[0.8, 1.2],
    rotation_range=5,
    shear_range=3, 
    horizontal_flip=False)

aug_videos = []
aug_labels = []

for vid, label in zip(X_train, y_train):
    seed = np.random.randint(1e6)

    frames = [datagen_2.random_transform(frame, seed=seed) for frame in vid]
    aug_videos.append(np.array(frames, dtype=np.float32))
    aug_labels.append(label)

aug_videos = np.array(aug_videos, dtype=np.float32)
aug_labels = np.array(aug_labels)


X_combined = np.concatenate([X_train, aug_videos], axis=0)
y_combined = np.concatenate([y_train, aug_labels], axis=0)
meta_combined = np.concatenate([meta_train, meta_train], axis=0)

# Shuffle again
X_combined, y_combined, meta_combined = shuffle(
    X_combined, y_combined, meta_combined, random_state=42)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils import shuffle

datagen_3 = ImageDataGenerator(
    width_shift_range=0.12,
    height_shift_range=0.12,
    zoom_range=0.12,
    brightness_range=[0.75, 1.25],
    rotation_range=8,
    shear_range=5, 
    horizontal_flip=False)

aug_videos = []
aug_labels = []

for vid, label in zip(X_train, y_train):
    seed = np.random.randint(1e6)

    frames = [datagen_3.random_transform(frame, seed=seed) for frame in vid]
    aug_videos.append(np.array(frames, dtype=np.float32))
    aug_labels.append(label)

aug_videos = np.array(aug_videos, dtype=np.float32)
aug_labels = np.array(aug_labels)


X_combined = np.concatenate([X_train, aug_videos], axis=0)
y_combined = np.concatenate([y_train, aug_labels], axis=0)
meta_combined = np.concatenate([meta_train, meta_train], axis=0)

In [ ]:
X_combined, y_combined, meta_combined = shuffle(
    X_combined, y_combined, meta_combined, random_state=42)

In [8]:
del aug_videos, aug_labels, meta_train, X_train, y_train, datagen
gc.collect()

0

In [9]:
print("Train augmented dataset shapes:")
print("X_combined:", X_combined.shape)
print("y_combined:", y_combined.shape)
print("meta_combined:", meta_combined.shape)

Train augmented dataset shapes:
X_combined: (2100, 24, 224, 224, 3)
y_combined: (2100,)
meta_combined: (2100, 5)


In [ ]:
np.savez_compressed(
    "../data/augmented_train_dataset.npz",
    X_train=X_combined,
    X_val = X_val,
    y_train=y_combined,
    y_val=y_val,
    meta_train=meta_combined,
    meta_val=meta_val)

In [ ]:
np.savez_compressed(
    "../data/augmented_train_dataset_2.npz",
    X_train=X_combined,
    X_val = X_val,
    y_train=y_combined,
    y_val=y_val,
    meta_train=meta_combined,
    meta_val=meta_val)

In [ ]:
np.savez_compressed(
    "../data/augmented_train_dataset_3.npz",
    X_train=X_combined,
    X_val = X_val,
    y_train=y_combined,
    y_val=y_val,
    meta_train=meta_combined,
    meta_val=meta_val)

In [ ]:
data = np.load("augmented_train_dataset.npz")
X_train = data["X_train"]
X_val = data["X_val"]
y_train = data["y_train"]
y_val = data["y_val"]
meta_train = data["meta_train"]
meta_val = data["meta_val"]

del data
gc.collect()

2314